## Basic trip moving cargo
In this notebook, we set up a basic simulation where one vessel moves over a 1D network path consisting of one edge. We add the HasContainer mixin to both the nodes of the network and the vessel. We created a simple customized mission, where the vessel sails back and forth loading/unloading until the destination site is full.

In [ ]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
from pyproj import Geod
from shapely.geometry import Point

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim

# OpenTNSim mixins and utils
from opentnsim.core import Identifiable, Movable, HasContainer, Locatable
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.utils import create_object, inspect_object
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.visualizations import plot_graph

# package(s) needed for inspecting the output
import pandas as pd
import numpy as np

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

#### 0. Create environment
We create the environment

In [ ]:
# start simpy environment
simulation_start = datetime.datetime(2024, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph
Next we create our graph. For this case we create a single edge of 100 km exactly.

In [ ]:
# initialize geodetic calculator with WGS84 ellipsoid
geod = Geod(ellps="WGS84")

# starting point (longitude, latitude)
lon0, lat0 = 0, 0

# compute the other point 100 km East from Point 0
lon1, lat1, _ = geod.fwd(lon0, lat0, 90, 100000) # East from point 1

# define nodes with their geographic coordinates, and capacity and level
node_info = {
    "0": {'position': (lon0, lat0), 'capacity': 100, 'level': 100},
    "1": {'position': (lon1, lat1), 'capacity': 100, 'level': 0},
}

Different from previous graphs, we now create our own special nodes using an object:

In [ ]:
# make your preferred Node class out of available mix-ins.
Node = create_object(
    "Node", 
    (
        Identifiable, # allows to give the object a name and a random ID, 
        Locatable,    # allows the object to have a location
        HasContainer, # allows the object to contain cargo
    ),
)

In [ ]:
# we can inspect our Node-object
inspect_object(Node, show_parameter_table=True)

In [ ]:
# create a directed graph
FG = nx.DiGraph()

# create sites from dict (we use the parameter table to add the required input) 
nodes = []
for name, node in node_info.items():
    data_node = {
        "env": env,
        "name": name,
        "geometry": Point(node['position'][0], node['position'][1]), 
        "capacity": node['capacity'],
        "level": node['level'],
    }
    nodes.append(Node(**data_node))
    
# add sites to graph
for node in nodes:
    FG.add_node(node.name, geometry=node.geometry, site=node)

In [ ]:
# create list of edges
edges = [("0", "1"), ("1", "0")] # bi-directional edge

# add edges
for edge in edges:
    FG.add_edge(edge[0], edge[1], weight=1)

In [ ]:
plot_graph(FG)

In [ ]:
# add graph to environment
env.graph = FG

#### 2. Create vessel
We create a vessel that can move cargo:

In [ ]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel", 
    (
        Identifiable, # allows to give the object a name and a random ID,
        Movable,      # allows the object to move, with a fixed speed, while logging this activity
        HasContainer, # allows the object to contain cargo
    ), 
)

We create a special mission for our vessel:

In [ ]:
def mission(env, vessel, origin, destination):
    """
    Method that defines the mission of the vessel. 
    
    In this case: 
        load - move - unload - move until the to_site is full
    """

    while True:
        # *** load ***
        current_node = '0'
        amount = np.min([vessel.container.capacity, env.graph.nodes['0']['site'].container.level])
        duration = amount * 1000
        
        vessel.log_entry_v0(
                    "Loading from node {} start".format(current_node),
                    vessel.env.now, 0, env.graph.nodes[current_node]['geometry'])
        
        yield env.graph.nodes['0']['site'].container.get(amount)
        yield vessel.container.put(amount)
        yield env.timeout(duration)
    
        vessel.log_entry_v0(
                    "Loading from node {} stop".format(current_node),
                    vessel.env.now, 0, env.graph.nodes[current_node]['geometry'])
            
        # *** move ***
        route = nx.dijkstra_path(env.graph, origin, destination)
        vessel.route = route
        while True:
            yield from vessel.move()
            
            if vessel.geometry == nx.get_node_attributes(FG, "geometry")[vessel.route[-1]]:
                break    
    
        # *** unload ***
        current_node = '1'
        amount = vessel.container.level
        duration = amount * 1000
    
        vessel.log_entry_v0(
                    "Unloading to node {} start".format(current_node),
                    vessel.env.now, 0, env.graph.nodes[current_node]['geometry'])
    
        yield vessel.container.get(amount)
        yield env.graph.nodes['1']['site'].container.put(amount)
        yield env.timeout(duration)
        
        vessel.log_entry_v0(
                    "Unloading to node {} stop".format(current_node),
                    vessel.env.now, 0, env.graph.nodes[current_node]['geometry'])
        
        # *** move ***
        vessel.route = nx.dijkstra_path(env.graph, destination, origin)    
        while True:
            yield from vessel.move()
            
            if vessel.geometry == nx.get_node_attributes(FG, "geometry")[vessel.route[-1]]:
                break

        if env.graph.nodes['1']['site'].container.level == env.graph.nodes['1']['site'].container.capacity:
            break

In [ ]:
# create vessel from dict 
data_vessel = {
    "env": env,                                   # needed for simpy simulation, left empty for now
    "name": "Vessel",                             # required by Identifiable
    "geometry": env.graph.nodes['0']['geometry'], # required by Locatable, left empty for now
    "route": [],                                  # required by Routeable, left empty for now
    "v": 1,                                       # required by Movable, 1 m/s to check if the distance is covered in the expected time
    "capacity": 30,                               # required by HasContainer
    "level": 0,                                   # required by HasContainer
}  
vessel = Vessel(**data_vessel)

# start the simulation
env.process(mission(env, vessel, '0', '1'))

In [ ]:
vessel.route

In [ ]:
vessel.edge_route

#### 3. Run simulation

In [ ]:
# start the simulation
env.run()

#### 4. Inspect output
We can now inspect  the simulation output by inspecting the _vessel.logbook_. Note that the _Log_ mix-in was included when we added _Movable_. The _vessel.logbook_ keeps track of the moving activities of the vessel. For each discrete event OpenTNSim logs an event message, the start/stop time and the location. The _vessel.logbook_ is of type dict. For convenient inspection it can be loaded into a Pandas dataframe. 

In [ ]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel.logbook)

print("'{}' logbook data:".format(vessel.name))  
print('')

display(df)

The inspection of the logbook data shows that Vessel moved from its origin (*Node 0*) to its destination (*Node 1*). The print statements show that the length of the route from *Node 0* to *Node 1* is exactly 100 km. At a given speed of 1 m/s the trip duration should be exactly 100000 seconds, as is indeed shown to be the case.

In [ ]:
df_eventtable = logbook2eventtable([vessel])
df_eventtable

In [ ]:
generate_vessel_gantt_chart(df_eventtable)

The inspection of the logbook data shows that the mission of the Vessel, resulted in four cycles where loading - sailing - unloading - sailing were executed in a sequence. The sites had a capacity of 100, and the Vessel a capacity of 30. It can be seen that the last cycle the loading and unloading take less time because the Vessel sailed only partially filled.